# RMS (pump-noise) campaign — analysis template

**What this is.** A copy-me notebook for an *RMS* campaign: the pump power is fixed and we sweep the **P1 polariser angle** to change the pump intensity noise $\sigma^2$, watching how g² and R respond. Each sub-folder of the data root is one angle.

**Model link.** Increasing $\sigma^2$ should push g² up ($g_n^{(2)}=1+K(n)^2\sigma^2$) while keeping $R\le 1$, dipping deepest near $\sigma^2=1/(K_mK_n)$. The angle is read from the acquisition's `rotation_stage`, so messy folder names don't matter.

In [ ]:
# --- bootstrap: make `src` importable and run from the project root ---
import sys, os
from pathlib import Path
ROOT = Path.cwd()
if not (ROOT / 'src').is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from src import Campaign, GridVisualizer, CampaignReport, model, config
config.apply_style(usetex=True)

## 1. Configure & load
Point `DATA_ROOT` at the RMS root (it contains one sub-folder per angle).

`K` sets the effective nonlinearity used for the normalised collapse. Default is the ideal $K(n)=n$; if you have a matching power scan, pass its measured $K$ instead, e.g. `K={3: 2.6, 5: 3.1}`.

In [ ]:
DATA_ROOT = 'data/Jun18/RMS'   # <-- edit me
HARMONICS = (3, 5)             # H4 was dead on Jun18; drop it
TAU_INT   = 4.0
K         = None              # e.g. {3: 2.6, 5: 3.1} from a power scan; None -> ideal K(n)=n

camp = Campaign.from_subfolders(DATA_ROOT, harmonics=HARMONICS, tau_in_ns=TAU_INT, K=K)
print(camp.runs[0].acquisition_summary())
camp.summary_table()

## 2. Overview vs P1 angle
g²(0), Cauchy-Schwarz R, singles and the normalised excess as the pump noise is varied.

In [ ]:
camp.plot_overview();

## 3. R stays classical
Zoom on R vs angle against the classical bound $R=1$ — the headline result.

In [ ]:
camp.plot_R(ylim=(0.9, 1.05));

## 4. Coherence & sweeps (overlaying angles)

In [ ]:
gv = GridVisualizer(camp.runs, labels=camp.labels, comparison_variable='P1 angle')
gv.plot_coherence(xlim=8.0, integration_window_ns=TAU_INT)
gv.plot_R(methods=['delay'], tau_min=0.3, tau_max=25, step=1.0);

## 5. Export the report

In [ ]:
CampaignReport(camp).export(matrices=True)